# Phase 12 — Controlled Model Challenger Benchmark

- **12A.** Experiment Contract & Safety
- **12B.** Champion Reproduction
- **12C.** Benchmark Utilities
- **12D.** Historical Controls
- **12E.** Focused 13–72 XGBoost
- **12F.** Focused XGBoost Variants
- **12G.** HistGradientBoosting
- **12H.** Random Forest
- **12I.** Stage-1 Leaderboard
- **12K.** Robustness Analysis
- **12L.** Validation Selection Freeze
- **12M.** One-Time Final Test
- **12N.** Final Model Decision
- **12O.** Production Impact Plan
- **12P.** Save Benchmark Artifacts

## **12A.** Experiment Contract & Safety Validation

Before training any challenger, this phase verifies that the benchmark is
operating on the exact artifacts and data contract used by the existing
production model.

This phase validates:

- required experiment artifacts
- Phase 2 and production feature-contract agreement
- ordered model features
- target and identifier columns
- train, validation, and test schemas
- frozen split row counts and timestamp boundaries
- complete 1–72 hour forecast-horizon coverage
- missing-value constraints
- duplicate reference/horizon constraints
- chronological and purge-boundary integrity
- production model metadata consistency
- local software environment and artifact fingerprints

The test split is accessed here only for structural contract validation.
No test predictions, target-based test metrics, or model-selection decisions
are performed in this phase.

No model is trained and no production artifact is modified.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
def find_project_root(
    start: Path | None = None,
) -> Path:
    """Locate the repository root from the current notebook environment."""

    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "data" / "training").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate the Pearls AQI Predictor project root."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "training"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Training data:", DATA_DIR)
print("Models:", MODEL_DIR)
print("Reports:", REPORT_DIR)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Training data: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training
Models: /home/riyan/Riyan/projects/pearls-aqi-predictor/models
Reports: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports


In [3]:
PHASE_2_FEATURE_CONTRACT_PATH = (
    DATA_DIR / "feature_columns.json"
)

PHASE_2_VALIDATION_REPORT_PATH = (
    DATA_DIR / "phase_2_validation_report.json"
)

TRAIN_DATASET_PATH = (
    DATA_DIR / "train_dataset.parquet"
)

VALIDATION_DATASET_PATH = (
    DATA_DIR / "validation_dataset.parquet"
)

TEST_DATASET_PATH = (
    DATA_DIR / "test_dataset.parquet"
)

PRODUCTION_MODEL_PATH = (
    MODEL_DIR / "best_model.joblib"
)

PRODUCTION_FEATURE_CONTRACT_PATH = (
    MODEL_DIR / "model_feature_columns.json"
)

PRODUCTION_METADATA_PATH = (
    MODEL_DIR / "model_metadata.json"
)

MODEL_SELECTION_REPORT_PATH = (
    MODEL_DIR / "model_selection_report.json"
)


REQUIRED_ARTIFACTS = {
    "phase_2_feature_contract": PHASE_2_FEATURE_CONTRACT_PATH,
    "phase_2_validation_report": PHASE_2_VALIDATION_REPORT_PATH,
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "production_model": PRODUCTION_MODEL_PATH,
    "production_feature_contract": PRODUCTION_FEATURE_CONTRACT_PATH,
    "production_metadata": PRODUCTION_METADATA_PATH,
    "model_selection_report": MODEL_SELECTION_REPORT_PATH,
}

In [4]:
artifact_status_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
        }
        for name, path in REQUIRED_ARTIFACTS.items()
    ]
)

display(artifact_status_df)

missing_artifacts = [
    name
    for name, path in REQUIRED_ARTIFACTS.items()
    if not path.exists()
]

assert not missing_artifacts, (
    "Required benchmark artifacts are missing: "
    f"{missing_artifacts}"
)

print("All required benchmark artifacts exist.")

,artifact,path,exists
0,phase_2_feature_contract,data/training/feature_columns.json,True
1,phase_2_validation_report,data/training/phase_2_validation_report.json,True
2,train_dataset,data/training/train_dataset.parquet,True
3,validation_dataset,data/training/validation_dataset.parquet,True
4,test_dataset,data/training/test_dataset.parquet,True
5,production_model,models/best_model.joblib,True
6,production_feature_contract,models/model_feature_columns.json,True
7,production_metadata,models/model_metadata.json,True
8,model_selection_report,models/model_selection_report.json,True


All required benchmark artifacts exist.


In [5]:
def load_json_object(
    path: Path,
) -> dict[str, Any]:
    """Load one JSON artifact and require a JSON object."""

    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    if not isinstance(payload, dict):
        raise ValueError(
            f"{path} must contain a JSON object."
        )

    return payload


phase_2_feature_contract = load_json_object(
    PHASE_2_FEATURE_CONTRACT_PATH
)

phase_2_validation_report = load_json_object(
    PHASE_2_VALIDATION_REPORT_PATH
)

production_feature_contract = load_json_object(
    PRODUCTION_FEATURE_CONTRACT_PATH
)

production_metadata = load_json_object(
    PRODUCTION_METADATA_PATH
)

model_selection_report = load_json_object(
    MODEL_SELECTION_REPORT_PATH
)

print("Experiment metadata loaded successfully.")

Experiment metadata loaded successfully.


In [6]:
MODEL_FEATURE_COLUMNS = list(
    phase_2_feature_contract["feature_columns"]
)

TARGET_COLUMN = str(
    phase_2_feature_contract["target_column"]
)

IDENTIFIER_COLUMNS = list(
    phase_2_feature_contract["identifier_columns"]
)

FORECAST_HORIZON_COLUMN = (
    "forecast_horizon_hours"
)

EXPECTED_HORIZONS = list(
    range(
        int(
            phase_2_feature_contract[
                "forecast_horizon_min"
            ]
        ),
        int(
            phase_2_feature_contract[
                "forecast_horizon_max"
            ]
        )
        + 1,
    )
)


print("Model features:", len(MODEL_FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Identifiers:", IDENTIFIER_COLUMNS)
print(
    "Forecast horizons:",
    EXPECTED_HORIZONS[0],
    "to",
    EXPECTED_HORIZONS[-1],
)

Model features: 56
Target: target_pm25_ug_m3
Identifiers: ['reference_time', 'target_time']
Forecast horizons: 1 to 72


In [7]:
production_feature_columns = list(
    production_feature_contract["feature_columns"]
)

metadata_feature_columns = list(
    production_metadata["ordered_feature_names"]
)

assert MODEL_FEATURE_COLUMNS == production_feature_columns, (
    "Phase 2 and production feature contracts differ."
)

assert MODEL_FEATURE_COLUMNS == metadata_feature_columns, (
    "Phase 2 features and production metadata features differ."
)

assert len(MODEL_FEATURE_COLUMNS) == 56

assert (
    production_feature_contract["feature_count"]
    == 56
)

assert (
    production_metadata["input_feature_count"]
    == 56
)

assert (
    TARGET_COLUMN
    == production_feature_contract["target_column"]
    == production_metadata["target_column"]
)

assert (
    IDENTIFIER_COLUMNS
    == production_feature_contract["identifier_columns"]
)

assert (
    FORECAST_HORIZON_COLUMN
    in MODEL_FEATURE_COLUMNS
)

assert TARGET_COLUMN not in MODEL_FEATURE_COLUMNS

for identifier_column in IDENTIFIER_COLUMNS:
    assert identifier_column not in MODEL_FEATURE_COLUMNS


future_pm25_features = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column.startswith("target_pm25")
]

assert not future_pm25_features, (
    "Future PM2.5-derived model inputs detected: "
    f"{future_pm25_features}"
)

print("Feature-contract agreement validated.")

Feature-contract agreement validated.


In [18]:
train_df = pd.read_parquet(
    TRAIN_DATASET_PATH
)

validation_df = pd.read_parquet(
    VALIDATION_DATASET_PATH
)

test_df = pd.read_parquet(
    TEST_DATASET_PATH
)


dataset_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "columns": len(train_df.columns),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "columns": len(validation_df.columns),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "columns": len(test_df.columns),
        },
    ]
)

display(dataset_summary_df)

,split,rows,columns
0,train,364798,59
1,validation,71256,59
2,test,76813,59


In [20]:
expected_splits = (
    phase_2_validation_report["splits"]
)

assert len(train_df) == int(
    expected_splits["train"]["rows"]
)

assert len(validation_df) == int(
    expected_splits["validation"]["rows"]
)

assert len(test_df) == int(
    expected_splits["test"]["rows"]
)


assert (
    train_df.columns.tolist()
    == validation_df.columns.tolist()
    == test_df.columns.tolist()
)

assert train_df.dtypes.equals(
    validation_df.dtypes
)

assert train_df.dtypes.equals(
    test_df.dtypes
)


required_columns = {
    *MODEL_FEATURE_COLUMNS,
    TARGET_COLUMN,
    *IDENTIFIER_COLUMNS,
}

missing_columns = sorted(
    required_columns.difference(
        train_df.columns
    )
)

assert not missing_columns, (
    f"Training schema is missing columns: {missing_columns}"
)

print("Frozen split sizes and schemas validated.")

Frozen split sizes and schemas validated.


In [21]:
def normalize_timestamp_columns(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Return a copy with identifier timestamps normalized to UTC."""

    result = dataframe.copy()

    for column in IDENTIFIER_COLUMNS:
        result[column] = pd.to_datetime(
            result[column],
            utc=True,
            errors="raise",
        )

    return result


train_checked_df = normalize_timestamp_columns(
    train_df
)

validation_checked_df = normalize_timestamp_columns(
    validation_df
)

test_checked_df = normalize_timestamp_columns(
    test_df
)

print("Timestamp columns normalized for contract checks.")

Timestamp columns normalized for contract checks.


In [22]:
split_frames = {
    "train": train_checked_df,
    "validation": validation_checked_df,
    "test": test_checked_df,
}


integrity_records = []

for split_name, dataframe in split_frames.items():
    missing_features = int(
        dataframe[
            MODEL_FEATURE_COLUMNS
        ]
        .isna()
        .sum()
        .sum()
    )

    missing_targets = int(
        dataframe[
            TARGET_COLUMN
        ].isna().sum()
    )

    duplicate_keys = int(
        dataframe.duplicated(
            subset=[
                "reference_time",
                FORECAST_HORIZON_COLUMN,
            ]
        ).sum()
    )

    horizons = sorted(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    integrity_records.append(
        {
            "split": split_name,
            "missing_features": missing_features,
            "missing_targets": missing_targets,
            "duplicate_reference_horizon_keys": duplicate_keys,
            "horizon_min": min(horizons),
            "horizon_max": max(horizons),
            "unique_horizons": len(horizons),
        }
    )

    assert missing_features == 0
    assert missing_targets == 0
    assert duplicate_keys == 0
    assert horizons == EXPECTED_HORIZONS


integrity_df = pd.DataFrame(
    integrity_records
)

display(integrity_df)

print("Dataset integrity validation passed.")

,split,missing_features,missing_targets,duplicate_reference_horizon_keys,horizon_min,horizon_max,unique_horizons
0,train,0,0,0,1,72,72
1,validation,0,0,0,1,72,72
2,test,0,0,0,1,72,72


Dataset integrity validation passed.


In [23]:
split_boundary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "reference_start": (
                train_checked_df["reference_time"].min()
            ),
            "reference_end": (
                train_checked_df["reference_time"].max()
            ),
            "target_start": (
                train_checked_df["target_time"].min()
            ),
            "target_end": (
                train_checked_df["target_time"].max()
            ),
        },
        {
            "split": "validation",
            "reference_start": (
                validation_checked_df["reference_time"].min()
            ),
            "reference_end": (
                validation_checked_df["reference_time"].max()
            ),
            "target_start": (
                validation_checked_df["target_time"].min()
            ),
            "target_end": (
                validation_checked_df["target_time"].max()
            ),
        },
        {
            "split": "test",
            "reference_start": (
                test_checked_df["reference_time"].min()
            ),
            "reference_end": (
                test_checked_df["reference_time"].max()
            ),
            "target_start": (
                test_checked_df["target_time"].min()
            ),
            "target_end": (
                test_checked_df["target_time"].max()
            ),
        },
    ]
)

display(split_boundary_df)

,split,reference_start,reference_end,target_start,target_end
0,train,2025-07-09 00:00:00+00:00,2026-03-18 11:00:00+00:00,2025-07-09 01:00:00+00:00,2026-03-21 11:00:00+00:00
1,validation,2026-03-21 12:00:00+00:00,2026-05-27 21:00:00+00:00,2026-03-21 13:00:00+00:00,2026-05-30 21:00:00+00:00
2,test,2026-05-30 22:00:00+00:00,2026-07-23 22:00:00+00:00,2026-05-30 23:00:00+00:00,2026-07-23 23:00:00+00:00


In [24]:
for split_name, dataframe in split_frames.items():
    expected = expected_splits[split_name]

    assert (
        dataframe["reference_time"].min()
        == pd.Timestamp(
            expected["reference_start"]
        )
    )

    assert (
        dataframe["reference_time"].max()
        == pd.Timestamp(
            expected["reference_end"]
        )
    )

    assert (
        dataframe["target_time"].min()
        == pd.Timestamp(
            expected["target_start"]
        )
    )

    assert (
        dataframe["target_time"].max()
        == pd.Timestamp(
            expected["target_end"]
        )
    )


assert (
    train_checked_df["target_time"].max()
    < validation_checked_df["reference_time"].min()
)

assert (
    validation_checked_df["target_time"].max()
    < test_checked_df["reference_time"].min()
)


for dataframe in split_frames.values():
    assert (
        dataframe["target_time"]
        > dataframe["reference_time"]
    ).all()


print("Chronological split and purge boundaries validated.")

Chronological split and purge boundaries validated.


In [25]:
EXPECTED_STRATEGY = (
    "hybrid_persistence_1_12_"
    "xgboost_shallower_13_72"
)

PERSISTENCE_MAX_HORIZON = 12


assert (
    production_metadata["selected_strategy"]
    == EXPECTED_STRATEGY
)

assert (
    model_selection_report["selected_strategy"]
    == EXPECTED_STRATEGY
)

assert (
    production_metadata["model_name"]
    == "xgboost_shallower"
)

assert (
    model_selection_report["selected_model"]
    == "xgboost_shallower"
)

assert (
    int(
        production_metadata[
            "routing"
        ]["persistence_max_horizon"]
    )
    == PERSISTENCE_MAX_HORIZON
)

assert (
    production_metadata[
        "routing"
    ]["horizons_1_to_12"]
    == "current_pm25_persistence"
)

assert (
    production_metadata[
        "routing"
    ]["horizons_13_to_72"]
    == "xgboost_shallower"
)


print("Production strategy metadata validated.")

Production strategy metadata validated.


In [26]:
current_environment = {
    "python": platform.python_version(),
    "pandas": version("pandas"),
    "numpy": version("numpy"),
    "scikit_learn": version("scikit-learn"),
    "xgboost": version("xgboost"),
    "joblib": version("joblib"),
}


production_environment = (
    production_metadata.get(
        "software_versions",
        {}
    )
)


environment_comparison_df = pd.DataFrame(
    [
        {
            "package": package,
            "current": current_environment.get(
                package
            ),
            "production_training": (
                production_environment.get(
                    package
                )
            ),
        }
        for package in current_environment
    ]
)

display(environment_comparison_df)

,package,current,production_training
0,python,3.12.3,3.12.3
1,pandas,2.3.3,3.0.5
2,numpy,2.2.6,2.5.1
3,scikit_learn,1.9.0,1.9.0
4,xgboost,3.3.0,3.3.0
5,joblib,1.5.3,None


In [27]:
def calculate_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """Calculate the SHA-256 checksum of a local artifact."""

    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


fingerprint_paths = {
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "phase_2_feature_contract": (
        PHASE_2_FEATURE_CONTRACT_PATH
    ),
    "production_feature_contract": (
        PRODUCTION_FEATURE_CONTRACT_PATH
    ),
    "production_model": PRODUCTION_MODEL_PATH,
    "production_metadata": PRODUCTION_METADATA_PATH,
}


artifact_fingerprints = {
    name: calculate_sha256(path)
    for name, path in fingerprint_paths.items()
}


fingerprint_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "sha256": checksum,
        }
        for name, checksum
        in artifact_fingerprints.items()
    ]
)

display(fingerprint_df)

,artifact,sha256
0,train_dataset,aa8bd58e0a1dc3dbc7e35c1dd4216d5400da22ef26d656...
1,validation_dataset,b30b259d1daeec75386a0b9dfb81fa8d90deaaa22d86e3...
2,test_dataset,0359ff647d1bfa3f021ca8e0ae6a2313a4831addb09a5d...
3,phase_2_feature_contract,21260e0ab94d9196ff5fa4d782517cf113d61f8eb578c2...
4,production_feature_contract,613b785c93a2c4135416906cb4a1c09b402c53df68674e...
5,production_model,9d1cbb032a6f376892c2ad047bbf93bf485d18b21b47d1...
6,production_metadata,11a398dec9abdc619d25db46b3065ae400db4cfb9df49b...


In [28]:
contract_validation_summary = {
    "status": "PASSED",
    "feature_count": len(
        MODEL_FEATURE_COLUMNS
    ),
    "target_column": TARGET_COLUMN,
    "identifier_columns": (
        IDENTIFIER_COLUMNS
    ),
    "forecast_horizon_min": min(
        EXPECTED_HORIZONS
    ),
    "forecast_horizon_max": max(
        EXPECTED_HORIZONS
    ),
    "train_rows": len(train_df),
    "validation_rows": len(
        validation_df
    ),
    "test_rows": len(test_df),
    "selected_strategy": (
        production_metadata[
            "selected_strategy"
        ]
    ),
    "production_model_name": (
        production_metadata["model_name"]
    ),
    "persistence_max_horizon": (
        PERSISTENCE_MAX_HORIZON
    ),
}


display(
    pd.Series(
        contract_validation_summary,
        name="value",
    ).to_frame()
)

print(
    "Phase 12A PASSED — frozen experiment "
    "contract is internally consistent."
)

,value
status,PASSED
feature_count,56
target_column,target_pm25_ug_m3
identifier_columns,"[reference_time, target_time]"
forecast_horizon_min,1
forecast_horizon_max,72
train_rows,364798
validation_rows,71256
test_rows,76813
selected_strategy,hybrid_persistence_1_12_xgboost_shallower_13_72


Phase 12A PASSED — frozen experiment contract is internally consistent.
